In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications, callbacks
import os

# ==========================================
# 1. 하이퍼파라미터 및 기본 설정
# ==========================================
BASE_DIR = './new_base_dir' # 생성해두신 이진 분류 폴더 경로로 맞춰주세요
BATCH_SIZE = 64
IMG_SIZE = (96, 96) # ★ 핵심: 48x48 원본을 96x96으로 업스케일링하여 로드

# ==========================================
# 2. 데이터셋 로드 및 최적화
# ==========================================
train_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'train'),
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    color_mode='rgb',
    label_mode='binary'
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'test'),
    shuffle=False, # 검증 데이터는 섞을 필요 없음
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    color_mode='rgb',
    label_mode='binary'
)

# 데이터 로드 병목 현상 방지
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

# ==========================================
# 3. 데이터 증강 (Overfitting 방지용)
# ==========================================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")

# ==========================================
# 4. 모델 아키텍처 설계
# ==========================================
# 4-1. 전처리 및 베이스 모델 (MobileNetV2)
preprocess_input = applications.mobilenet_v2.preprocess_input
base_model = applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), 
    include_top=False, 
    weights='imagenet'
)
base_model.trainable = False # 1단계 학습을 위해 몸통 전체 동결

# 4-2. 분류기(헤드) 부착
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)       # 1. 증강
x = preprocess_input(x)             # 2. 스케일링 (-1 ~ 1)
x = base_model(x, training=False)   # 3. 특징 추출 (BatchNormalization 동결 유지)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)          # ★ 핵심: 과적합 방지를 위해 50% 노드 끄기
outputs = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs, outputs)

# ==========================================
# 5. [1단계] Warm-up 학습 (분류기만 학습)
# ==========================================
print("\n--- Phase 1: 분류기(Top layer) 학습 시작 ---")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

history_warmup = model.fit(
    train_dataset,
    epochs=10, # 분류기만 가볍게 10 에포크 학습
    validation_data=validation_dataset
)

# ==========================================
# 6. [2단계] Fine-tuning (미세 조정)
# ==========================================
print("\n--- Phase 2: 미세 조정(Fine-tuning) 시작 ---")

# 몸통 동결 해제
base_model.trainable = True

# ★ 핵심: 하위 130개 레이어는 꽁꽁 얼려두고, 최상위 레이어 일부만 학습 허용
freeze_until = 130
for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

# 아주 작은 학습률로 컴파일 (기존 지식 파괴 방지)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

# 콜백 설정 (조기 종료 및 학습률 조정)
callbacks_list = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1) # 성능 정체 시 학습률 반으로 줄임
]

# 이어서 학습
history_fine = model.fit(
    train_dataset,
    epochs=30, # 조기 종료가 있으니 넉넉하게 설정
    validation_data=validation_dataset,
    callbacks=callbacks_list
)

print("\n🎉 모든 학습이 완료되었습니다!")